In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("loan_approval_data.csv")

In [ ]:
df.drop("Applicant_ID", axis=1) #dropping applicant id which is unnecessary

In [ ]:
df.head()  # irst 5 rows of your DataFrame. [For 10 rows df.head(10)]
df.info()  # 1000 inputs and 20 columns (info)
df.describe()

Predict a loan approval system (Yes/No). SO It become a Binary Classification (A model that chooses between two categories Yes/No). {Log Reg/ kNN/ Naive bayes} we will check which one gives best prediction. Do data cleaning (handle data cleaning).

Handle Data Handling. Most ML algorithms cannot handle NaN(“Not a Number” = missing data) values. If we try to train without fixing them → error.

For Num (age, loan amt) = Avg val {mean, median (middle value)}.
For Catagorical val (Gender, typically objects) = Mode

In [ ]:
catagorical_col = df.select_dtypes(include=["object"]).columns
numerical_col = df.select_dtypes(include=["number"]).columns

# .select_dtypes() selects columns based data type. "object" text/string data. "number" gives float/int types

In [ ]:
from sklearn.impute import SimpleImputer  # fill missing NaN values automatically.

num_imp = SimpleImputer(strategy="mean")
df[numerical_col] = num_imp.fit_transform(df[numerical_col])

# “If something is missing, replace it with the average (mean) of that column.” fit() calculates the mean (using num_imp strategy) of each numerical column. transform() replaces missing values with that mean.

# But when its catagorical (object) col, mean/median doesnt make sense ( “Male” + “Female” / 2 makes no sense) to replace with. We use Most frequent instead or Unknown (better than NaN).

cat_imp = SimpleImputer(strategy="most_frequent")
df[catagorical_col] = cat_imp.fit_transform(df[catagorical_col])

EDA (Exploratory Data Analysis). We are exploring the data and analyzing.

In [ ]:
#How balanced our classes are?

classes_count = df["Loan_Approved"].value_counts()
plt.pie(classes_count, labels=["No", "Yes"], autopct= "%1.1f%%")
plt.title("Is loan approved or not?")

In [ ]:
sns.boxplot(
    data= df,
    x= "Loan_Approved",
    y= "Applicant_Income"
)

We will remove outliers (A data point that is significantly different from the majority of the data). cause an outlier can skew the mean of data, lead to biased predictions (Ex: If most loan amounts are 100–500k, a 10M loan will make the model think higher loans are normal). we can detect it using visual outliers (Boxplot/Histogram/Scatter plot) or statistical methods (Z-score)

In [ ]:
# Credit score with loan approved.

sns.histplot(data=df, x="Credit_Score", hue="Loan_Approved", bins=20, multiple="dodge")


In [ ]:
sns.histplot(
    data=df, x="Applicant_Income", hue="Loan_Approved", bins=20, multiple="dodge"
)

We have Object (texts/catagories) and most of our Machine Learning algorithms (like LogisticRegression, RandomForest, LinearRegression) cannot work directly with text/categorical data.

Computers only understand numbers, not words.

Label Encoding: Assigns a unique number to each category. {Male = 1. Female = 0} 

One-Hot Encoding: Creates a binary column for each category. {nominal data = no order.}


In [ ]:
df.info()

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

le = LabelEncoder()
df["Education_Level"] = le.fit_transform(
    df["Education_Level"]
)  # encode col Edu_Lev using label encod(le) and store in a new column called "Education_Level" in my DataFrame.
df["Loan_Purpose"] = le.fit_transform(df["Loan_Purpose"])

In [ ]:
cols = [
    "Employment_Status",
    "Marital_Status",
    "Loan_Purpose",
    "Property_Area",
    "Education_Level",
    "Gender",
    "Employer_Category",
    "Loan_Approved",
]

ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(
    encoded, columns=ohe.get_feature_names_out(cols), index=df.index
)
df = pd.concat([df.drop(columns=cols), encoded_df], axis=1)

# Now use df (not encoded_df) which has ALL features
X = df.drop(columns=[col for col in df.columns if "Loan_Approved" in col])
y = df[[col for col in df.columns if "Loan_Approved" in col][0]]

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Now scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_test_scaled #Now we have all scaled values.

Train & Evalution

In [ ]:
# Model-1 -> Logestic reg

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

log_model = LogisticRegression()
log_model.fit(X_train_scaled, y_train)
y_prediction = log_model.predict(X_test_scaled)

# Evalution [Precision, Accuracy, Recall, F1]

print("Precision: ", precision_score(y_test, y_prediction))
print("Recall: ", recall_score(y_test, y_prediction))
print("F1 Score: ", f1_score(y_test, y_prediction))
print("Accuracy: ", accuracy_score(y_test, y_prediction))
print("Confusion: ", confusion_matrix(y_test, y_prediction))

In [ ]:
# Model-2 -> kNN

from sklearn.neighbors import KNeighborsClassifier

kNN_model = KNeighborsClassifier(n_neighbors=5)
kNN_model.fit(X_train_scaled, y_train)
y_prediction = kNN_model.predict(X_test_scaled)

# Evalution [Precision, Accuracy, Recall, F1]

print("Precision: ", precision_score(y_test, y_prediction))
print("Recall: ", recall_score(y_test, y_prediction))
print("F1 Score: ", f1_score(y_test, y_prediction))
print("Accuracy: ", accuracy_score(y_test, y_prediction))
print("Confusion: ", confusion_matrix(y_test, y_prediction))

We have worse performance in kNN than Logistics

In [ ]:
# Model-3 -> Naive Bayes

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)
y_prediction = nb_model.predict(X_test_scaled)

# Evalution [Precision, Accuracy, Recall, F1]

print("Precision: ", precision_score(y_test, y_prediction))
print("Recall: ", recall_score(y_test, y_prediction))
print("F1 Score: ", f1_score(y_test, y_prediction))
print("Accuracy: ", accuracy_score(y_test, y_prediction))
print("Confusion: ", confusion_matrix(y_test, y_prediction))